In [8]:
# HomeGuard Smart Home Monitoring System
# Author: Barsha
# Description: Simulates home sensors (temperature, smoke, motion, door)
#              and triggers alerts depending on the system mode.

import random
from datetime import datetime

In [9]:
def create_sensor(sensor_id, location, sensor_type):
    """Return a dictionary that describes one sensor."""
    return {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "current_value": None,
    }


def create_alert(sensor, severity, category, message):
    """Return a dictionary that describes one alert."""
    return {
        "sensor_id": sensor["id"],
        "location": sensor["location"],
        "severity": severity,
        "category": category,
        "message": message,
        "timestamp": datetime.now().strftime("%H:%M:%S"),
    }


def initialize_sensors():
    """Create the list of sensors for the Peterson vacation home."""
    return [
        create_sensor("T1", "Living Room", "temperature"),
        create_sensor("T2", "Bedroom", "temperature"),
        create_sensor("S1", "Kitchen", "smoke"),
        create_sensor("M1", "Living Room", "motion"),
        create_sensor("M2", "Bedroom", "motion"),
        create_sensor("M3", "Kitchen", "motion"),
        create_sensor("D1", "Front Door", "door"),
        create_sensor("D2", "Back Door", "door"),
    ]

sensors = initialize_sensors()
for sensor in sensors:
    print(sensor)  

{'id': 'T1', 'location': 'Living Room', 'type': 'temperature', 'current_value': None}
{'id': 'T2', 'location': 'Bedroom', 'type': 'temperature', 'current_value': None}
{'id': 'S1', 'location': 'Kitchen', 'type': 'smoke', 'current_value': None}
{'id': 'M1', 'location': 'Living Room', 'type': 'motion', 'current_value': None}
{'id': 'M2', 'location': 'Bedroom', 'type': 'motion', 'current_value': None}
{'id': 'M3', 'location': 'Kitchen', 'type': 'motion', 'current_value': None}
{'id': 'D1', 'location': 'Front Door', 'type': 'door', 'current_value': None}
{'id': 'D2', 'location': 'Back Door', 'type': 'door', 'current_value': None}


In [10]:
def check_reading(sensor, system_mode):
    """Return (severity, category, message) if the reading is abnormal,
    otherwise return None."""
    value = sensor["current_value"]
    sensor_type = sensor["type"]

    # Safety checks: these run in every mode
    if sensor_type == "smoke":
        if value > 50:
            return ("CRITICAL", "SAFETY", f"Smoke detected ({value} ppm)")
    elif sensor_type == "temperature":
        if value < 35:
            return ("CRITICAL", "SAFETY", f"Temperature below 35F - frozen pipe risk ({value}F)")
        elif value > 95:
            return ("CRITICAL", "SAFETY", f"Temperature above 95F - equipment failure ({value}F)")

    # Security checks: only when the house is in AWAY mode
    if system_mode == "AWAY":
        if sensor_type == "motion" and value == 1:
            return ("HIGH", "SECURITY", "Motion detected while away")
        if sensor_type == "door" and value == 1:
            return ("HIGH", "SECURITY", "Door opened while away")

    # Comfort check: only when someone is home
    if system_mode == "HOME" and sensor_type == "temperature":
        if value < 65 or value > 75:
            return ("LOW", "COMFORT", f"Temperature outside comfort range 65-75F ({value}F)")

    # Nothing wrong
    return None

    sensors = initialize_sensors()

# Set some readings by hand
sensors[0]["current_value"] = 100   # Living Room temperature
sensors[3]["current_value"] = 1     # Living Room motion
sensors[2]["current_value"] = 5     # Kitchen smoke

print(check_reading(sensors[0], "HOME"))   # too hot
print(check_reading(sensors[3], "AWAY"))   # motion while away
print(check_reading(sensors[3], "HOME"))   # motion while home
print(check_reading(sensors[2], "HOME"))   # smoke is low

('CRITICAL', 'SAFETY', 'Temperature above 95F - equipment failure (100F)')
('HIGH', 'SECURITY', 'Motion detected while away')
None
None


In [13]:
event_log = []


def log_event(text):
    """Record an event with a timestamp."""
    entry = f"[{datetime.now().strftime('%H:%M:%S')}] {text}"
    event_log.append(entry)


def generate_reading(sensor):
    """Create a realistic random reading for a sensor and store it."""
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        value = round(random.gauss(70, 12), 1)       # degrees F
    elif sensor_type == "smoke":
        if random.random() < 0.05:                   # 5% chance of smoke
            value = random.randint(60, 150)
        else:
            value = random.randint(0, 10)            # ppm
    elif sensor_type == "motion":
        value = 1 if random.random() < 0.25 else 0   # 25% chance
    elif sensor_type == "door":
        value = 1 if random.random() < 0.15 else 0   # 15% chance
    else:
        value = None

    sensor["current_value"] = value
    return value


def process_reading(sensor, system_mode):
    """Generate a reading, log it, and return an alert if one is needed."""
    generate_reading(sensor)
    log_event(f"{sensor['location']} {sensor['type']} = {sensor['current_value']}")

    result = check_reading(sensor, system_mode)
    if result is None:
        return None

    severity, category, message = result
    return create_alert(sensor, severity, category, message)


def trigger_alert(alert):
    """Display an alert and log it."""
    print(f"  [{alert['severity']}] {alert['category']}: "
          f"{alert['message']} @ {alert['location']}")
    log_event(f"ALERT {alert['severity']} - {alert['message']}")


def check_multiple_triggers(alerts, system_mode):
    """If 2+ security alerts happen in one cycle, return a break-in alert."""
    security_alerts = [a for a in alerts if a["category"] == "SECURITY"]

    if system_mode == "AWAY" and len(security_alerts) >= 2:
        return {
            "sensor_id": "MULTI",
            "location": "Multiple",
            "severity": "CRITICAL",
            "category": "SECURITY",
            "message": f"{len(security_alerts)} sensors triggered at once - possible break-in",
            "timestamp": datetime.now().strftime("%H:%M:%S"),
        }
    return None

sensors = initialize_sensors()
alerts = []

for sensor in sensors:
    alert = process_reading(sensor, "AWAY")
    print(sensor["location"], sensor["type"], sensor["current_value"])
    if alert:
        trigger_alert(alert)
        alerts.append(alert)

break_in = check_multiple_triggers(alerts, "AWAY")
if break_in:
    trigger_alert(break_in)

print()
print("Events logged:", len(event_log))  

Living Room temperature 75.6
Bedroom temperature 54.3
Kitchen smoke 2
Living Room motion 1
  [HIGH] SECURITY: Motion detected while away @ Living Room
Bedroom motion 0
Kitchen motion 0
Front Door door 1
  [HIGH] SECURITY: Door opened while away @ Front Door
Back Door door 0
  [CRITICAL] SECURITY: 2 sensors triggered at once - possible break-in @ Multiple

Events logged: 11


In [17]:
class Sensor:
    def __init__(self, sensor_id, location, sensor_type):
        self.sensor_id = sensor_id
        self.location = location
        self.sensor_type = sensor_type
        self.current_value = None
        self.minutes_open = 0      # used by door sensors

    def read(self):
        """Generate a realistic random reading and store it."""
        if self.sensor_type == "temperature":
            self.current_value = round(random.gauss(70, 12), 1)
        elif self.sensor_type == "smoke":
            if random.random() < 0.05:
                self.current_value = random.randint(60, 150)
            else:
                self.current_value = random.randint(0, 10)
        elif self.sensor_type == "motion":
            self.current_value = 1 if random.random() < 0.25 else 0
        elif self.sensor_type == "door":
            if self.current_value == 1:
                # an open door tends to stay open
                self.current_value = 1 if random.random() < 0.8 else 0
            else:
                self.current_value = 1 if random.random() < 0.15 else 0
            # count how many readings in a row the door has been open
            if self.current_value == 1:
                self.minutes_open += 1
            else:
                self.minutes_open = 0
        return self.current_value

    def evaluate(self, system_mode):
        """Return (severity, category, message) if abnormal, else None."""
        value = self.current_value

        # Safety checks: every mode
        if self.sensor_type == "smoke":
            if value > 50:
                return ("CRITICAL", "SAFETY", f"Smoke detected ({value} ppm)")
        elif self.sensor_type == "temperature":
            if value < 35:
                return ("CRITICAL", "SAFETY", f"Temperature below 35F - frozen pipe risk ({value}F)")
            elif value > 95:
                return ("CRITICAL", "SAFETY", f"Temperature above 95F - equipment failure ({value}F)")

        # Security checks: AWAY mode only
        if system_mode == "AWAY":
            if self.sensor_type == "motion" and value == 1:
                return ("HIGH", "SECURITY", "Motion detected while away")
            if self.sensor_type == "door" and value == 1:
                return ("HIGH", "SECURITY", "Door opened while away")

        # Unusual pattern: door left open for more than 5 minutes
        if self.sensor_type == "door" and self.minutes_open > 5:
            return ("LOW", "COMFORT", f"Door left open for {self.minutes_open} minutes")

        # Comfort check: HOME mode only
        if system_mode == "HOME" and self.sensor_type == "temperature":
            if value < 65 or value > 75:
                return ("LOW", "COMFORT", f"Temperature outside comfort range 65-75F ({value}F)")

        return None

    def isAbnormal(self, system_mode):
        """True if the current reading should raise an alert."""
        return self.evaluate(system_mode) is not None

    def reset(self):
        """Clear the stored reading."""
        self.current_value = None
        self.minutes_open = 0

def create_alert(sensor, severity, category, message):
    return {
        "sensor_id": sensor.sensor_id,
        "location": sensor.location,
        "severity": severity,
        "category": category,
        "message": message,
    }


def initialize_sensors():
    return [
        Sensor("T1", "Living Room", "temperature"),
        Sensor("T2", "Bedroom", "temperature"),
        Sensor("S1", "Kitchen", "smoke"),
        Sensor("M1", "Living Room", "motion"),
        Sensor("M2", "Bedroom", "motion"),
        Sensor("M3", "Kitchen", "motion"),
        Sensor("D1", "Front Door", "door"),
        Sensor("D2", "Back Door", "door"),
    ]


def process_reading(sensor, system_mode):
    sensor.read()
    log_event(f"{sensor.location} {sensor.sensor_type} = {sensor.current_value}")

    result = sensor.evaluate(system_mode)
    if result is None:
        return None

    severity, category, message = result
    return create_alert(sensor, severity, category, message)
    
s = Sensor("T1", "Living Room", "temperature")
print(s.read())                 # a random temperature
print(s.isAbnormal("HOME"))     # True or False, depending on that value
s.current_value = 100
print(s.isAbnormal("HOME"))     # True (too hot)
s.reset()
print(s.current_value)          # None

door = Sensor("D1", "Front Door", "door")
door.current_value = 1
door.minutes_open = 6
print(door.evaluate("HOME"))    # door left open alert

70.1
False
True
None
('LOW', 'COMFORT', 'Door left open for 6 minutes')
